# 40 — Spectra, spectrograms and resolution checks

Compare density-normalized Hann-Welch and equal-weight DPSS spectra. If a selected trace has an policy-selected noise/event pair, use those exact intervals and identical estimator settings. Otherwise show an explicitly labelled **whole-record exploratory comparison**, never invented noise/event windows.

Gap-aware Hann spectrograms use 1.28, 2.56 and 5.12-second windows where available, plus a normalized DPSS comparison. A short step smooths the display; it does not increase independent resolution. Longer windows improve frequency discrimination at the cost of timing. DPSS smoothing bandwidth differs from its frequency-bin spacing. No zero-padding is used.

Fixed/detected windows follow the policy saved by notebook 30; individual manual review is optional.


In [ ]:
from pathlib import Path
import sys
_candidates = [Path.cwd(), Path.cwd() / '20_analysis', *Path.cwd().parents]
ANALYSIS_DIR = next((p for p in _candidates if (p / 'analysis_workflow.py').is_file()), None)
if ANALYSIS_DIR is None:
    raise FileNotFoundError('Start in 20_analysis or its project root, beside analysis_workflow.py.')
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))
from analysis_workflow import load_context, describe_context
from IPython.display import display, Image, Markdown
import matplotlib.pyplot as plt
import pandas as pd
ctx = load_context(ANALYSIS_DIR)
display(describe_context(ctx))


In [ ]:
from analysis_workflow import build_inventory, spectral_examples
inventory, _ = build_inventory(ctx)
display(inventory.loc[inventory.is_waveform, ['trace_key', 'trace_id', 'duration_s', 'qc_flags']])
EXAMPLE_TRACE_KEYS = []  # explicit keys preferred; default: first waveform in each selected record
if not EXAMPLE_TRACE_KEYS:
    EXAMPLE_TRACE_KEYS = inventory.loc[inventory.is_waveform].groupby('record_id', sort=False).head(1).trace_key.tolist()
for key in EXAMPLE_TRACE_KEYS[:ctx['config']['plots']['max_examples']]:
    paths, figures = spectral_examples(ctx, key)
    for path in paths:
        display(Image(filename=str(path)))
    for fig in figures:
        plt.close(fig)


## Interpreting the comparison

All PSDs use count²/Hz, with dB relative to **1 count²/Hz**. Spectrogram panels share fixed configured colour limits; saturated colours do not mean waveform clipping. Fourier window lengths/steps are rounded to whole sample counts and actual values are saved. Rejected FFT windows remain missing. Numerical arrays and FLOVOpy spectral-evolution tables accompany the figures.

FLOVOpy's older strongest-trace multitaper plotting function uses a legacy power scaling, so this notebook uses SciPy density-normalized taper periodograms instead of treating that display as a calibrated PSD. Multitaper here is equal-weight DPSS, not adaptive weighting. Estimator disagreement should be investigated rather than choosing the sharper-looking image.

## SAC/reassignment follow-up

[SAC's spectrogram documentation](https://ds.iris.edu/files/sac-manual/commands/spectrogram.html) distinguishes PDS, maximum-entropy and maximum-likelihood estimators, with a separate model-order parameter. Those parametric methods and reassignment are **not implemented as validated PSD alternatives in this first sequence**. Before interpreting narrow features, compare stability across window lengths, DPSS bandwidth and (for future parametric estimates) model order on synthetic signals and held-out real records. No claim of reproducing SAC or SWARM is made here.